改进点：采用交叉验证优化阈值，采用v1的逻辑回归模型

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [2]:
def quick_and_correct_optimization():
    train = pd.read_csv('train_diabetes.csv')
    test = pd.read_csv('test_diabetes.csv')
    features = ['Age', 'Gender', 'BMI', 'Glucose', 'BloodPressure',
                'SkinThickness', 'Insulin', 'HbA1c', 'ExerciseHours',
                'DietScore', 'Smoking', 'AlcoholConsumption', 'FamilyHistory']
    X_train = train[features]
    y_train = train['Diabetes']
    X_test = test[features]

    # 填充缺失值
    for col in features:
        if X_train[col].isnull().any():
            median_val = X_train[col].median()
            X_train[col] = X_train[col].fillna(median_val)
            X_test[col] = X_test[col].fillna(median_val)

    # 分割出验证集
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )

    # 训练模型
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)

    # 优化阈值（在验证集上）
    val_proba = lr.predict_proba(X_val)[:, 1]
    best_thresh = 0.5
    best_acc = 0

    for thresh in np.arange(0.3, 0.71, 0.01):
        val_pred = (val_proba >= thresh).astype(int)
        acc = (val_pred == y_val).mean()
        if acc > best_acc:
            best_acc = acc
            best_thresh = thresh

    print(f"最佳阈值: {best_thresh:.3f} (验证集准确率: {best_acc:.2%})")

    # 用全部数据重新训练
    lr_full = LogisticRegression(max_iter=1000, random_state=42)
    lr_full.fit(X_train, y_train)

    # 预测测试集
    test_proba = lr_full.predict_proba(X_test)[:, 1]
    predictions = (test_proba >= best_thresh).astype(int)

    # 保存结果
    result = pd.DataFrame({
        'PatientID': test['PatientID'],
        'Diabetes': predictions
    })

    result.to_csv('Diabetes_prediction_v6.csv', index=False)
    print(f"预测完成！保存到Diabetes_prediction_v6.csv")

    return result, best_thresh

In [5]:
# 验证准确率

from sklearn.metrics import accuracy_score
# 读取文件
pred = pd.read_csv('Diabetes_prediction_v6.csv')
true = pd.read_csv('test_answer.csv')

# 合并并计算准确率
merged = pred.merge(true, on='PatientID')
accuracy = accuracy_score(merged['Diabetes_x'], merged['Diabetes_y'])

print(f"模型准确率: {accuracy*100:.2f}%")

模型准确率: 54.00%
